In [ ]:

from pathlib import Path
import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().resolve().parent
DATA_PATH = PROJECT_ROOT / "data/processed/cereal_priced.parquet"

df = pd.read_parquet(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

display(df.head())


Rows: 4,707,776
Columns: 14


,UPC,STORE,WEEK,MOVE,QTY,PRICE,SALE,PROFIT,UNIT_PRICE,REVENUE,UNIT_AAC_PROXY,GROSS_PROFIT_PROXY,DESCRIP,SIZE
0,317,51,372,1,1,26.020,<NA>,62.52,26.020,26.02,9.752296,16.267704,$TONY THE TIGER T-SH,ASST
1,317,74,367,1,1,19.060,<NA>,48.84,19.060,19.06,9.751096,9.308904,$TONY THE TIGER T-SH,ASST
2,317,93,364,2,1,1.475,L,12.76,1.475,2.95,1.286790,0.376420,$TONY THE TIGER T-SH,ASST
3,317,104,383,1,1,3.170,<NA>,-7.57,3.170,3.17,3.409969,-0.239969,$TONY THE TIGER T-SH,ASST
4,317,111,381,1,1,10.460,<NA>,6.78,10.460,10.46,9.750812,0.709188,$TONY THE TIGER T-SH,ASST


In [2]:

# Cell 2 — Product catalog inspection

catalog_path = PROJECT_ROOT / "data/raw/upccer.csv"

catalog = pd.read_csv(
    catalog_path,
    encoding="cp1252",
    dtype={"UPC": "string"}
)

catalog.columns = catalog.columns.str.strip().str.upper()
catalog["UPC"] = catalog["UPC"].str.strip()

print("Catalog products:", len(catalog))
print("Cleaned products:", df["UPC"].nunique())

print("\nCOMMODITY CODE DISTRIBUTION")
display(catalog["COM_CODE"].value_counts(dropna=False))

print("\nFIRST 15 CATALOG ENTRIES")
display(
    catalog[
        ["UPC", "DESCRIP", "SIZE", "COM_CODE"]
    ].head(15)
)

# Find the one product absent after cleaning.
missing_products = catalog.loc[
    ~catalog["UPC"].isin(df["UPC"].unique())
]

print("\nPRODUCTS MISSING AFTER CLEANING")
display(
    missing_products[
        ["UPC", "DESCRIP", "SIZE", "COM_CODE"]
    ]
)


Catalog products: 490
Cleaned products: 489

COMMODITY CODE DISTRIBUTION


COM_CODE
311    490
Name: count, dtype: int64


FIRST 15 CATALOG ENTRIES


,UPC,DESCRIP,SIZE,COM_CODE
0,317,$TONY THE TIGER T-SH,ASST,311
1,1313000002,NAB SHREDDED WHEAT,10 OZ,311
2,1313000005,NABISCO SHREDDED WHE,15 OZ,311
3,1313000032,NABISCO 100% BRAN,17 OZ,311
4,1313000050,NAB SPOON SIZE SHRED,23.45O,311
5,1313000054,NAB SPOON SIZE SHRED,17.2 O,311
6,1313000148,NABISCO TEAM FLAKES,13 OZ,311
7,1313000698,NABISCO CREAM OF RIC,14 OZ,311
8,1313001041,NABISCO OAT BRAN RAI,15.5 O,311
9,1313001059,NABISCO WHEAT N BRAN,18 OZ,311



PRODUCTS MISSING AFTER CLEANING


,UPC,DESCRIP,SIZE,COM_CODE
236,3000043316,QUAKER CLUB PACK CAP,2/20 O,311


In [3]:

# Cell 3 — Product-level performance

product_summary = (
    df.groupby("UPC", as_index=False)
    .agg(
        description=("DESCRIP", "first"),
        size=("SIZE", "first"),
        total_units=("MOVE", "sum"),
        observations=("MOVE", "size"),
        stores=("STORE", "nunique"),
        weeks=("WEEK", "nunique"),
        distinct_prices=("UNIT_PRICE", "nunique"),
        median_price=("UNIT_PRICE", "median"),
    )
)

product_summary = product_summary.merge(
    catalog[["UPC", "COM_CODE"]],
    on="UPC",
    how="left",
    validate="one_to_one"
)

product_summary = product_summary.sort_values(
    "total_units", ascending=False
)

display(product_summary.head(15))


,UPC,description,size,total_units,observations,stores,weeks,distinct_prices,median_price,COM_CODE
71,1600066610,CHEERIOS,15 OZ,2011819,29679,93,366,226,3.25,311
243,3800000120,KELLOGGS CORN FLAKES,18 OZ,1830189,29787,93,366,180,2.15,311
69,1600066510,CHEERIOS,10 OZ,1560241,29379,93,366,159,2.35,311
70,1600066590,HONEY NUT CHEERIOS,14 OZ,1512370,29537,93,366,249,3.19,311
269,3800001520,KELLOGG'S FROSTED FL,20 OZ,1322703,29690,93,366,218,3.49,311
268,3800001510,KELLOGG'S FROSTED FL,15 OZ,1287272,29654,93,366,221,2.82,311
278,3800001720,KELLOGGS FRUIT LOOPS,15 OZ,1208611,29724,93,366,218,3.35,311
241,3800000110,KELLOGGS CORN FLAKES,12 OZ,1167781,29512,93,366,99,1.73,311
272,3800001611,KELLOGGS SPECIAL K,12 OZ,1077255,29771,93,366,133,2.99,311
248,3800000520,KELLOGGS RICE KRISPI,15 OZ,1063040,14752,86,201,123,2.45,311


In [4]:

# Cell 4 — Top-selling products

top15 = (
    product_summary.head(15)
    .sort_values("total_units")
    .copy()
)

top15["product_label"] = (
    top15["description"].str.strip()
    + " | "
    + top15["UPC"]
)

fig = px.bar(
    top15,
    x="total_units",
    y="product_label",
    orientation="h",
    title="PriceWise | Top 15 Products by Units Sold",
    labels={
        "total_units": "Total Units Sold",
        "product_label": "Product",
    },
    hover_data=[
        "stores",
        "weeks",
        "distinct_prices",
        "median_price",
    ],
)

fig.update_layout(
    height=650,
    yaxis_title=None,
    xaxis_title="Total Units Sold",
)

fig.show()


In [5]:

# Cell 5 — Inspect promotion indicators

display(
    df["SALE"]
    .value_counts(dropna=False)
    .rename_axis("SALE_CODE")
    .reset_index(name="OBSERVATIONS")
)


,SALE_CODE,OBSERVATIONS
0,<NA>,4350772
1,B,251546
2,S,91077
3,G,11058
4,C,3322
5,L,1


In [7]:

# Cell 6 — Product and store selection

TARGET_UPC = "1600066610"  # Cheerios 15 oz

cheerios = df.loc[
    df["UPC"] == TARGET_UPC,
    [
        "UPC", "STORE", "WEEK", "MOVE",
        "UNIT_PRICE", "SALE"
    ]
].copy()

# Which stores have the most usable weeks?
store_summary = (
    cheerios.groupby("STORE")
    .agg(
        observed_weeks=("WEEK", "nunique"),
        price_levels=("UNIT_PRICE", "nunique"),
        total_units=("MOVE", "sum"),
    )
    .sort_values(
        ["observed_weeks", "price_levels"],
        ascending=False
    )
)

display(store_summary.head(10))

# Choose the store with the most coverage.
STORE_ID = int(store_summary.index[0])

store_df = (
    cheerios.loc[cheerios["STORE"] == STORE_ID]
    .sort_values("WEEK")
    .copy()
)

store_df["SALE_LABEL"] = (
    store_df["SALE"].fillna("Unmarked")
)

print(f"Selected store: {STORE_ID}")
print(f"Observed weeks: {store_df['WEEK'].nunique()}")
print(f"Different prices: {store_df['UNIT_PRICE'].nunique()}")
print(f"Median price: ${store_df['UNIT_PRICE'].median():.2f}")

display(store_df.head())


,observed_weeks,price_levels,total_units
STORE,,,
101,365,46,27206
83,365,41,26188
78,364,43,30993
84,364,39,25337
76,363,43,19386
80,363,40,39935
132,363,40,28936
73,363,35,30028
103,362,51,20684


Selected store: 101
Observed weeks: 365
Different prices: 46
Median price: $3.20


,UPC,STORE,WEEK,MOVE,UNIT_PRICE,SALE,SALE_LABEL
774033,1600066610,101,1,112,2.68,<NA>,Unmarked
774034,1600066610,101,2,136,2.68,<NA>,Unmarked
774035,1600066610,101,3,177,2.68,<NA>,Unmarked
774036,1600066610,101,4,129,2.68,<NA>,Unmarked
774037,1600066610,101,5,138,2.68,<NA>,Unmarked


In [8]:

# Cell 7 — Price vs. observed weekly demand

fig = px.scatter(
    store_df,
    x="UNIT_PRICE",
    y="MOVE",
    color="SALE_LABEL",
    hover_data=["WEEK", "STORE"],
    title=f"Cheerios 15 oz | Price vs Sales | Store {STORE_ID}",
    labels={
        "UNIT_PRICE": "Unit Price ($)",
        "MOVE": "Weekly Units Sold",
        "SALE_LABEL": "Recorded Promotion",
    },
)

fig.update_traces(
    marker=dict(size=9, opacity=0.7)
)

fig.update_layout(
    height=550,
    template="plotly_white",
)

fig.show()


In [9]:

# Cell 8 — Price and demand over time

import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=[
        "Weekly Unit Price",
        "Weekly Units Sold"
    ],
)

fig.add_trace(
    go.Scatter(
        x=store_df["WEEK"],
        y=store_df["UNIT_PRICE"],
        mode="lines+markers",
        name="Unit Price",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=store_df["WEEK"],
        y=store_df["MOVE"],
        mode="lines+markers",
        name="Units Sold",
    ),
    row=2,
    col=1,
)

fig.update_yaxes(
    title_text="Price ($)",
    row=1, col=1,
)

fig.update_yaxes(
    title_text="Units",
    row=2, col=1,
)

fig.update_xaxes(
    title_text="Historical Week",
    row=2, col=1,
)

fig.update_layout(
    title=f"Cheerios 15 oz | Store {STORE_ID}",
    height=700,
    template="plotly_white",
    hovermode="x unified",
)

fig.show()

# Cell 8 — Price and demand over time

import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=[
        "Weekly Unit Price",
        "Weekly Units Sold"
    ],
)

fig.add_trace(
    go.Scatter(
        x=store_df["WEEK"],
        y=store_df["UNIT_PRICE"],
        mode="lines+markers",
        name="Unit Price",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=store_df["WEEK"],
        y=store_df["MOVE"],
        mode="lines+markers",
        name="Units Sold",
    ),
    row=2,
    col=1,
)

fig.update_yaxes(
    title_text="Price ($)",
    row=1, col=1,
)

fig.update_yaxes(
    title_text="Units",
    row=2, col=1,
)

fig.update_xaxes(
    title_text="Historical Week",
    row=2, col=1,
)

fig.update_layout(
    title=f"Cheerios 15 oz | Store {STORE_ID}",
    height=700,
    template="plotly_white",
    hovermode="x unified",
)

fig.show()


In [10]:

# Cell 9 — Investigate week coverage

all_weeks = set(df["WEEK"].unique())
store_weeks = set(store_df["WEEK"].unique())

first_week = int(df["WEEK"].min())
last_week = int(df["WEEK"].max())

expected_range = set(
    range(first_week, last_week + 1)
)

missing_everywhere = sorted(
    expected_range - all_weeks
)

missing_in_store = sorted(
    all_weeks - store_weeks
)

print("Global week range:", first_week, "to", last_week)

print(
    "Weeks absent from the cleaned dataset:",
    len(missing_everywhere)
)
print(missing_everywhere)

print(
    "\nWeeks present elsewhere but absent in Store 101:",
    len(missing_in_store)
)
print(missing_in_store)


Global week range: 1 to 399
Weeks absent from the cleaned dataset: 33
[219, 262, 263, 264, 265, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 370, 371]

Weeks present elsewhere but absent in Store 101: 1
[np.int64(30)]


In [11]:

# Cell 10 — Compare demand across price bands

analysis = store_df.copy()

analysis["RECORDED_PROMO"] = (
    analysis["SALE_LABEL"] != "Unmarked"
)

analysis["PRICE_BAND"] = pd.qcut(
    analysis["UNIT_PRICE"],
    q=4,
    duplicates="drop"
)

price_summary = (
    analysis.groupby("PRICE_BAND", observed=True)
    .agg(
        observations=("MOVE", "size"),
        median_price=("UNIT_PRICE", "median"),
        mean_units=("MOVE", "mean"),
        median_units=("MOVE", "median"),
        promo_fraction=("RECORDED_PROMO", "mean"),
    )
    .reset_index()
)

price_summary["recorded_promo_pct"] = (
    price_summary["promo_fraction"] * 100
).round(1)

price_summary["PRICE_BAND"] = (
    price_summary["PRICE_BAND"].astype(str)
)

display(price_summary.round(2))

fig = px.bar(
    price_summary,
    x="PRICE_BAND",
    y="median_units",
    hover_data=[
        "observations",
        "median_price",
        "mean_units",
        "recorded_promo_pct",
    ],
    title="Cheerios | Typical Weekly Sales by Price Band",
    labels={
        "PRICE_BAND": "Observed Unit Price Band ($)",
        "median_units": "Median Weekly Units Sold",
    },
)

fig.update_layout(
    template="plotly_white",
    height=500,
)

fig.show()


,PRICE_BAND,observations,median_price,mean_units,median_units,promo_fraction,recorded_promo_pct
0,"(1.0290000000000001, 3.09]",149,2.90,109.53,57.0,0.17,17.4
1,"(3.09, 3.2]",34,3.19,97.29,47.0,0.12,11.8
2,"(3.2, 3.35]",135,3.28,41.63,37.0,0.0,0.0
3,"(3.35, 3.59]",47,3.59,41.66,38.0,0.04,4.3


In [12]:

# Cell 11 — Week-over-week price and sales changes

transitions = (
    store_df.sort_values("WEEK")
    .reset_index(drop=True)
    .copy()
)

transitions["week_gap"] = (
    transitions["WEEK"].diff()
)

transitions["price_change_pct"] = (
    transitions["UNIT_PRICE"]
    .pct_change(fill_method=None) * 100
)

transitions["units_change_pct"] = (
    transitions["MOVE"]
    .pct_change(fill_method=None) * 100
)

# Only consecutive calendar weeks.
# Only price moves of at least 5%.
changes = transitions.loc[
    (transitions["week_gap"] == 1)
    & (transitions["price_change_pct"].abs() >= 5)
].copy()

changes["direction"] = (
    changes["price_change_pct"].map(
        lambda x: "Price cut" if x < 0
        else "Price increase"
    )
)

print("Price-change events:", len(changes))

display(
    changes.groupby("direction")
    .agg(
        events=("WEEK", "size"),
        median_unit_change_pct=(
            "units_change_pct", "median"
        ),
    )
)

fig = px.scatter(
    changes,
    x="price_change_pct",
    y="units_change_pct",
    color="direction",
    hover_data=[
        "WEEK",
        "UNIT_PRICE",
        "MOVE",
        "SALE_LABEL",
    ],
    title="Cheerios | Consecutive-Week Price & Sales Changes",
    labels={
        "price_change_pct": "Price Change (%)",
        "units_change_pct": "Unit Sales Change (%)",
        "direction": "Price Movement",
    },
)

fig.add_hline(y=0, line_dash="dash")
fig.add_vline(x=0, line_dash="dash")

fig.update_layout(
    template="plotly_white",
    height=550,
)

fig.show()


Price-change events: 45


,events,median_unit_change_pct
direction,,
Price cut,22,665.497076
Price increase,23,-84.523810


In [13]:

# Cell 12 — PriceWise product scope

TARGET_PRODUCTS = {
    "1600066610": "Cheerios 15 oz",
    "3800000120": "Corn Flakes 18 oz",
}

focus_df = df.loc[
    df["UPC"].isin(TARGET_PRODUCTS.keys())
].copy()

focus_df["PRODUCT"] = (
    focus_df["UPC"].map(TARGET_PRODUCTS)
)

# Enforce our expected project scope.
assert focus_df["UPC"].nunique() == 2
assert len(focus_df) == 59_466
assert focus_df["PRODUCT"].notna().all()

print("FOCUSED PRICEWISE DATASET")
print("Observations:", f"{len(focus_df):,}")
print("Products:", focus_df["UPC"].nunique())
print("Stores:", focus_df["STORE"].nunique())

display(
    focus_df.groupby("PRODUCT")
    .agg(
        observations=("MOVE", "size"),
        stores=("STORE", "nunique"),
        weeks=("WEEK", "nunique"),
        median_price=("UNIT_PRICE", "median"),
        median_units=("MOVE", "median"),
    )
)


FOCUSED PRICEWISE DATASET
Observations: 59,466
Products: 2
Stores: 93


,observations,stores,weeks,median_price,median_units
PRODUCT,,,,,
Cheerios 15 oz,29679,93,366,3.25,38.0
Corn Flakes 18 oz,29787,93,366,2.15,27.0


In [14]:

# Cell 13 — Compare coverage in Store 101

store_101 = focus_df.loc[
    focus_df["STORE"] == 101
].copy()

comparison = (
    store_101.groupby("PRODUCT")
    .agg(
        observed_weeks=("WEEK", "nunique"),
        price_levels=("UNIT_PRICE", "nunique"),
        median_price=("UNIT_PRICE", "median"),
        median_units=("MOVE", "median"),
        total_units=("MOVE", "sum"),
    )
)

display(comparison)

fig = px.box(
    store_101,
    x="PRODUCT",
    y="UNIT_PRICE",
    points=False,
    title="PriceWise | Price Distribution by Product — Store 101",
    labels={
        "PRODUCT": "Product",
        "UNIT_PRICE": "Observed Unit Price ($)",
    },
)

fig.update_layout(
    template="plotly_white",
    height=480,
)

fig.show()


,observed_weeks,price_levels,median_price,median_units,total_units
PRODUCT,,,,,
Cheerios 15 oz,365,46,3.20,45.0,27206
Corn Flakes 18 oz,365,43,2.05,34.0,25893


In [15]:

# Cell 14 — Save the focused modeling candidates

FOCUS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cereal_focus.parquet"
)

focus_df.to_parquet(
    FOCUS_PATH,
    index=False,
)

print(f"Saved {len(focus_df):,} observations")
print("Location:", FOCUS_PATH)


Saved 59,466 observations
Location: /home/amir/Projects/Full/PriceWise_Starter/data/processed/cereal_focus.parquet


In [16]:

# Cell 15 — Historical price support and candidate bounds

import numpy as np

support = (
    focus_df
    .groupby(["UPC", "PRODUCT", "STORE"], as_index=False)
    .agg(
        observed_weeks=("WEEK", "nunique"),
        price_levels=("UNIT_PRICE", "nunique"),
        p10_price=(
            "UNIT_PRICE",
            lambda x: x.quantile(0.10)
        ),
        p90_price=(
            "UNIT_PRICE",
            lambda x: x.quantile(0.90)
        ),
    )
)

# Latest observed price for each product and store.
latest = (
    focus_df
    .sort_values(["UPC", "STORE", "WEEK"])
    .groupby(["UPC", "STORE"], sort=False)
    .tail(1)
    [
        ["UPC", "STORE", "WEEK", "UNIT_PRICE"]
    ]
    .rename(columns={
        "WEEK": "last_observed_week",
        "UNIT_PRICE": "last_observed_price",
    })
)

support = support.merge(
    latest,
    on=["UPC", "STORE"],
    how="left",
    validate="one_to_one",
)

# Proposed exploratory guardrails: historical support
# intersected with a +/-10% change from last price.
support["candidate_min"] = np.maximum(
    support["p10_price"],
    support["last_observed_price"] * 0.90,
)

support["candidate_max"] = np.minimum(
    support["p90_price"],
    support["last_observed_price"] * 1.10,
)

support["range_available"] = (
    support["candidate_min"] <= support["candidate_max"]
)

print("PRODUCT-STORE COMBINATIONS")
print(len(support))

print("\nCOMBINATIONS WITH NONEMPTY DIAGNOSTIC RANGE")
print(support["range_available"].sum())

print("\nSTORE 101 — PRICE SUPPORT")
display(
    support.loc[
        support["STORE"] == 101,
        [
            "PRODUCT",
            "observed_weeks",
            "price_levels",
            "last_observed_week",
            "last_observed_price",
            "p10_price",
            "p90_price",
            "candidate_min",
            "candidate_max",
            "range_available",
        ]
    ].round(3)
)


PRODUCT-STORE COMBINATIONS
186

COMBINATIONS WITH NONEMPTY DIAGNOSTIC RANGE
186

STORE 101 — PRICE SUPPORT


,PRODUCT,observed_weeks,price_levels,last_observed_week,last_observed_price,p10_price,p90_price,candidate_min,candidate_max,range_available
53,Cheerios 15 oz,365,46,399,3.35,2.9,3.59,3.015,3.59,True
146,Corn Flakes 18 oz,365,43,399,2.85,1.9,2.84,2.565,2.84,True


In [17]:

# Cell 16 — Chronological split inspection

weeks = sorted(focus_df["WEEK"].unique())

train_end = int(len(weeks) * 0.70)
val_end = int(len(weeks) * 0.85)

train_weeks = set(weeks[:train_end])
val_weeks = set(weeks[train_end:val_end])
test_weeks = set(weeks[val_end:])

splits = {
    "Train": focus_df.loc[
        focus_df["WEEK"].isin(train_weeks)
    ],
    "Validation": focus_df.loc[
        focus_df["WEEK"].isin(val_weeks)
    ],
    "Test": focus_df.loc[
        focus_df["WEEK"].isin(test_weeks)
    ],
}

summary = []

for name, subset in splits.items():
    summary.append({
        "split": name,
        "rows": len(subset),
        "weeks": subset["WEEK"].nunique(),
        "first_week": subset["WEEK"].min(),
        "last_week": subset["WEEK"].max(),
        "products": subset["UPC"].nunique(),
        "stores": subset["STORE"].nunique(),
    })

split_summary = pd.DataFrame(summary)

assert len(train_weeks & val_weeks) == 0
assert len(train_weeks & test_weeks) == 0
assert len(val_weeks & test_weeks) == 0

assert sum(len(x) for x in splits.values()) == len(focus_df)

display(split_summary)


,split,rows,weeks,first_week,last_week,products,stores
0,Train,41570,256,1,257,2,86
1,Validation,8934,55,258,342,2,84
2,Test,8962,55,343,399,2,87


In [18]:

# Cell 17 — Leakage-safe historical feature engineering

import numpy as np
import pandas as pd

# Confirm the grain of our dataset.
KEY = ["UPC", "STORE", "WEEK"]

assert not focus_df.duplicated(KEY).any(), (
    "Duplicate product-store-week records found"
)

# Each group represents one product in one store.
engineered = (
    focus_df
    .sort_values(KEY)
    .copy()
)

groups = engineered.groupby(
    ["UPC", "STORE"],
    sort=False
)

# -----------------------------------------
# A. Past observations only
# -----------------------------------------

engineered["PREV_WEEK"] = (
    groups["WEEK"].shift(1)
)

engineered["WEEK_4_BACK"] = (
    groups["WEEK"].shift(4)
)

engineered["LAG_1_UNITS"] = (
    groups["MOVE"].shift(1)
)

engineered["LAG_1_PRICE"] = (
    groups["UNIT_PRICE"].shift(1)
)

# Average of the four PREVIOUS observed rows.
# We will verify calendar continuity below.
engineered["AVG_UNITS_PREV_4W"] = (
    groups["MOVE"].transform(
        lambda s: (
            s.shift(1)
             .rolling(4, min_periods=4)
             .mean()
        )
    )
)

# -----------------------------------------
# B. Verify four consecutive calendar weeks
# -----------------------------------------

engineered["HAS_4_WEEK_HISTORY"] = (
    engineered["WEEK"]
    - engineered["WEEK_4_BACK"]
).eq(4)

# -----------------------------------------
# C. Candidate-price features
# -----------------------------------------

engineered["CANDIDATE_PRICE"] = (
    engineered["UNIT_PRICE"]
)

engineered["PRICE_CHANGE_PCT"] = (
    100 * (
        engineered["CANDIDATE_PRICE"]
        / engineered["LAG_1_PRICE"]
        - 1
    )
)

# -----------------------------------------
# D. Promotion scenario context
# -----------------------------------------

engineered["PROMO_CONTEXT"] = (
    engineered["SALE"]
    .astype("string")
    .fillna("UNMARKED")
)

# -----------------------------------------
# E. Approximate 52-week seasonal cycle
# -----------------------------------------

season_position = (
    (engineered["WEEK"] - 1) % 52
)

engineered["WEEK_SIN"] = np.sin(
    2 * np.pi * season_position / 52
)

engineered["WEEK_COS"] = np.cos(
    2 * np.pi * season_position / 52
)

# -----------------------------------------
# F. Select observations with valid history
# -----------------------------------------

model_rows = engineered.loc[
    engineered["HAS_4_WEEK_HISTORY"]
].copy()

FEATURE_COLUMNS = [
    "UPC",
    "STORE",
    "CANDIDATE_PRICE",
    "LAG_1_PRICE",
    "LAG_1_UNITS",
    "AVG_UNITS_PREV_4W",
    "PRICE_CHANGE_PCT",
    "WEEK_SIN",
    "WEEK_COS",
    "PROMO_CONTEXT",
]

TARGET_COLUMN = "MOVE"

assert model_rows[FEATURE_COLUMNS].notna().all().all()
assert model_rows[TARGET_COLUMN].notna().all()

print("PRICEWISE | FEATURE ENGINEERING")
print("=" * 40)

print(f"Input rows:     {len(focus_df):,}")
print(f"Eligible rows:  {len(model_rows):,}")
print(f"History gaps:   {len(focus_df) - len(model_rows):,}")

print("\nFIRST FEATURE ROWS")

display(
    model_rows[
        [
            "PRODUCT",
            "STORE",
            "WEEK",
            "CANDIDATE_PRICE",
            "LAG_1_PRICE",
            "LAG_1_UNITS",
            "AVG_UNITS_PREV_4W",
            "PRICE_CHANGE_PCT",
            "MOVE",
        ]
    ].head(10).round(2)
)


PRICEWISE | FEATURE ENGINEERING
Input rows:     59,466
Eligible rows:  53,647
History gaps:   5,819

FIRST FEATURE ROWS


,PRODUCT,STORE,WEEK,CANDIDATE_PRICE,LAG_1_PRICE,LAG_1_UNITS,AVG_UNITS_PREV_4W,PRICE_CHANGE_PCT,MOVE
755763,Cheerios 15 oz,2,5,2.90,2.9,90.0,80.50,0.00,60
755764,Cheerios 15 oz,2,6,2.90,2.9,60.0,74.50,0.00,40
755765,Cheerios 15 oz,2,7,2.90,2.9,40.0,68.50,0.00,54
755766,Cheerios 15 oz,2,8,2.90,2.9,54.0,61.00,0.00,67
755767,Cheerios 15 oz,2,9,2.90,2.9,67.0,55.25,0.00,41
755768,Cheerios 15 oz,2,10,2.90,2.9,41.0,50.50,0.00,37
755769,Cheerios 15 oz,2,11,2.90,2.9,37.0,49.75,0.00,30
755770,Cheerios 15 oz,2,12,2.90,2.9,30.0,43.75,0.00,36
755771,Cheerios 15 oz,2,13,2.90,2.9,36.0,36.00,0.00,39
755772,Cheerios 15 oz,2,14,3.04,2.9,39.0,35.50,4.83,42


In [19]:

# Cell 18 — Assign chronological splits

TRAIN_LAST_WEEK = 257
VALIDATION_LAST_WEEK = 342

model_rows["SPLIT"] = pd.cut(
    model_rows["WEEK"],
    bins=[
        -np.inf,
        TRAIN_LAST_WEEK,
        VALIDATION_LAST_WEEK,
        np.inf,
    ],
    labels=["Train", "Validation", "Test"],
)

train = model_rows.loc[
    model_rows["SPLIT"] == "Train"
].copy()

validation = model_rows.loc[
    model_rows["SPLIT"] == "Validation"
].copy()

test = model_rows.loc[
    model_rows["SPLIT"] == "Test"
].copy()

# -----------------------------------------
# Verify our time boundaries
# -----------------------------------------

assert train["WEEK"].max() <= 257

assert validation["WEEK"].min() > 257
assert validation["WEEK"].max() <= 342

assert test["WEEK"].min() > 342

assert (
    len(train) + len(validation) + len(test)
    == len(model_rows)
)

# A retained row must have a true previous week.
assert (
    model_rows["WEEK"] - model_rows["PREV_WEEK"]
).eq(1).all()

# -----------------------------------------
# Inspect the final feature split
# -----------------------------------------

split_summary = pd.DataFrame([
    {
        "Split": name,
        "Rows": len(frame),
        "First week": frame["WEEK"].min(),
        "Last week": frame["WEEK"].max(),
        "Stores": frame["STORE"].nunique(),
    }
    for name, frame in [
        ("Train", train),
        ("Validation", validation),
        ("Test", test),
    ]
])

display(split_summary)

# -----------------------------------------
# Find new stores in evaluation periods
# -----------------------------------------

known_stores = set(train["STORE"])

for name, frame in [
    ("Validation", validation),
    ("Test", test),
]:
    unseen_stores = (
        set(frame["STORE"]) - known_stores
    )

    print(
        f"{name} stores absent from training:",
        sorted(unseen_stores)
    )

# -----------------------------------------
# Define X and y (no training yet)
# -----------------------------------------

X_train = train[FEATURE_COLUMNS].copy()
y_train = train[TARGET_COLUMN].copy()

X_val = validation[FEATURE_COLUMNS].copy()
y_val = validation[TARGET_COLUMN].copy()

X_test = test[FEATURE_COLUMNS].copy()
y_test = test[TARGET_COLUMN].copy()

print("\nML INPUTS AND TARGET")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

display(X_train.head())
display(y_train.head())


,Split,Rows,First week,Last week,Stores
0,Train,37912,5,257,86
1,Validation,7578,258,342,84
2,Test,8157,343,399,87


Validation stores absent from training: []
Test stores absent from training: [135, 140, 141, 142, 143, 144, 146]

ML INPUTS AND TARGET
X_train: (37912, 10)
y_train: (37912,)


,UPC,STORE,CANDIDATE_PRICE,LAG_1_PRICE,LAG_1_UNITS,AVG_UNITS_PREV_4W,PRICE_CHANGE_PCT,WEEK_SIN,WEEK_COS,PROMO_CONTEXT
755763,1600066610,2,2.9,2.9,90.0,80.50,0.0,0.464723,0.885456,UNMARKED
755764,1600066610,2,2.9,2.9,60.0,74.50,0.0,0.568065,0.822984,UNMARKED
755765,1600066610,2,2.9,2.9,40.0,68.50,0.0,0.663123,0.748511,UNMARKED
755766,1600066610,2,2.9,2.9,54.0,61.00,0.0,0.748511,0.663123,UNMARKED
755767,1600066610,2,2.9,2.9,67.0,55.25,0.0,0.822984,0.568065,UNMARKED


755763    60
755764    40
755765    54
755766    67
755767    41
Name: MOVE, dtype: int64

In [20]:

# Cell 19 — Feature eligibility by product

eligibility = (
    engineered.groupby("PRODUCT", as_index=False)
    .agg(
        original_rows=("MOVE", "size"),
        eligible_rows=("HAS_4_WEEK_HISTORY", "sum"),
    )
)

eligibility["eligibility_pct"] = (
    100
    * eligibility["eligible_rows"]
    / eligibility["original_rows"]
)

display(eligibility.round(2))

fig = px.bar(
    eligibility,
    x="PRODUCT",
    y="eligibility_pct",
    text="eligibility_pct",
    title="PriceWise | Four-Week History Availability",
    labels={
        "PRODUCT": "Product",
        "eligibility_pct": "Eligible Observations (%)",
    },
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
)

fig.update_layout(
    template="plotly_white",
    yaxis_range=[0, 105],
    height=430,
)

fig.show()


,PRODUCT,original_rows,eligible_rows,eligibility_pct
0,Cheerios 15 oz,29679,26771,90.20
1,Corn Flakes 18 oz,29787,26876,90.23


In [38]:

# Cell 20 — Verify reusable features
import sys
import os

# Add the src directory (one level up from notebooks) to the path
sys.path.append(os.path.abspath("../src"))

from pricewise.data.features import (
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    build_feature_table,
)

production_rows = build_feature_table(focus_df)

# The prototype and production implementation
# should contain exactly the same eligible rows.
assert len(production_rows) == len(model_rows)
assert len(production_rows) == 53_647

# Compare all model inputs and the target.
comparison_columns = [
    "UPC", "STORE", "WEEK",
    *[
        column
        for column in FEATURE_COLUMNS
        if column not in ("UPC", "STORE")
    ],
    TARGET_COLUMN,
]

def ordered(frame):
    return (
        frame[comparison_columns]
        .sort_values(["UPC", "STORE", "WEEK"])
        .reset_index(drop=True)
    )

pd.testing.assert_frame_equal(
    ordered(model_rows),
    ordered(production_rows),
    check_dtype=False,
    rtol=1e-12,
    atol=1e-12,
)

print("PASS: Notebook and production features match")
print("Rows:", len(production_rows))
print("Features:", len(FEATURE_COLUMNS))

display(
    production_rows[
        ["PRODUCT", "STORE", "WEEK"]
        + FEATURE_COLUMNS[2:6]
        + ["MOVE"]
    ].head()
)


PASS: Notebook and production features match
Rows: 53647
Features: 10


,PRODUCT,STORE,WEEK,CANDIDATE_PRICE,LAG_1_PRICE,LAG_1_UNITS,AVG_UNITS_PREV_4W,MOVE
755763,Cheerios 15 oz,2,5,2.9,2.9,90.0,80.50,60
755764,Cheerios 15 oz,2,6,2.9,2.9,60.0,74.50,40
755765,Cheerios 15 oz,2,7,2.9,2.9,40.0,68.50,54
755766,Cheerios 15 oz,2,8,2.9,2.9,54.0,61.00,67
755767,Cheerios 15 oz,2,9,2.9,2.9,67.0,55.25,41
